<a href="https://colab.research.google.com/github/cbonnin88/Python-For-Product/blob/main/DPM_projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Product Specification & Feature Simulation: Dynamic Carbon Emissions Alerts**

## **Context & Domain**
- Product: Enterprise Carbon Accounting & Compliance Platform

- Feature: Real-time Anomaly Detection & Operational Alerting

- Target User: Sustainability Officers & Facility Operations Managers at manufacturing corporations



### **The User Problem**
Sustainability officers are currently drowning in raw carbon emissions data. While our platform logs hourly $CO_2$ outputs perfectly, users are suffering from alert fatigue because of static thresholds

Manufacturing facilities have natural operational cycles; a factory naturally emits more during a Tuesday day-shift than a Sunday night-shift. Setting a hard, fixed ceiling triggers false alarms during peak operational hours and misses critical, slow-burning anomalies during low-cycle periods. If the alerts aren't context-aware, users will mute them, leading to unaddressed compliance breaches and hidden carbon tax penalties.

The Product Hypothesis: **If we introduce a dynamic threshold based on a 7-day rolling average ($168\text{ hours}$), then we can automatically filter out expected operational peaks while accurately catching true operational anomalies.**

We believe this approach will reduce false-positive alert volume by at least 35%, increasing alert actionability and keeping facility emissions safely within regional compliance bands without requiring manual configuration from the user.

Next Steps: This mini-simulation validates the baseline algorithm mechanics using Polars. To transition this feature into a production-ready engineering requirement, our next data discovery sprint must address:


1. **Baseline Seasonality Adjustments:** A flat 7-day rolling window does not account for monthly or quarterly manufacturing cyclicality. We need to evaluate a rolling window paired with day-of-week weights (e.g., comparing Tuesday performance against the mean of the past 4 Tuesdays).

2. **Alert Threshold Personalization:** The $1.5\times$ multiplier used in this prototype is a arbitrary heuristic. We need to run a sensitivity analysis against historical telemetry data to find the optimal statistical standard deviation ($\sigma$) that minimizes noise across diverse asset types (heavy industrial vs. light logistics).

3. **Data Pipeline Latency:** Polars processes this locally in microseconds, but a production deployment requires streaming analytics. We must verify if our data architecture can support low-latency window aggregations on continuous streaming sensor data via Kafka or Flink without spiking cloud computing costs.

In [ ]:
import polars as pl
import plotly.express as px
import numpy as np

In [ ]:
# 1. Simulating hourly factory emissions data over a month

df_product = pl.DataFrame({
    'timestamp': pl.datetime_range(start=pl.datetime(2026,1,1), end=pl.datetime(2026,1,31),interval='1h',eager=True),
    'co2_kg': pl.Series(np.random.rand(721)) * 500 + 100
})

In [ ]:
# 2. DPM feature Logic: Calculate a 7-day rolling average threshold
alert_simulation_df = df_product.with_columns([
    pl.col('co2_kg').rolling_mean(window_size=168).alias('7d_rolling_avg')
]).with_columns([
    # Define alert if current emission exceed 1.5x the rolling average
    (pl.col('co2_kg') > (pl.col('7d_rolling_avg') * 1.5)).alias('trigger_alert')
])

In [ ]:
# 3. Product Metric: Calculate total alerts generated to see if its too noisy
total_alerts = alert_simulation_df.filter(pl.col('trigger_alert') == True).shape[0]
print(f'Product Simulation Result: Threshold generates {total_alerts} alerts per month.')

Product Simulation Result: Threshold generates 80 alerts per month.


In [ ]:
# 4. Interactive Ploty Chart to present to stakeholders
fig = px.line(
    alert_simulation_df.to_pandas(),
    x='timestamp',
    y=['co2_kg','7d_rolling_avg'],
    title='Feature Simulation: Carbon Emissions Alerts'
)

alerts = alert_simulation_df.filter(pl.col('trigger_alert')==True)
fig.add_scatter(
    x=alerts['timestamp'],
    y=alerts['co2_kg'],
    mode='markers',
    name='Alert Triggered'
)

# **Product Specification & Proof of Concept: Microgrid Power Source Optimization**


## 📋 Context & Domain
- Product: Smart Microgrid Energy Management System (EMS)

- Feature: Automated Clean-Energy Dispatch Switching

- Target User: Microgrid Operators, Commercial & Industrial (C&I) Facility Managers, and Renewable Asset Engineers



## 🔍 The User Problem
Microgrid operators manage facilities powered by a mix of localized renewable energy (like onsite solar arrays) and the traditional, fossil-fuel-heavy utility grid. Currently, switching between these power sources is either handled manually or dictated by static, time-of-day schedules.

This creates massive operational inefficiency:

- On heavily overcast days, static schedules shift the load to solar arrays when solar irradiance is too low, causing voltage drops, battery drain, or unexpected brownouts.

- Conversely, on exceptionally clear mornings, the system remains hooked to the expensive, high-carbon utility grid long after the solar arrays have reached peak capacity, bleeding money and wasting clean energy.

- Operators need an intelligent, automated switcher that dynamically responds to real-time solar intensity rather than sticking to a rigid, blind clock.

## 💡 The Product Hypothesis
Ifwe build an automated dispatch switcher governed by live solar irradiance readings—routing power to the solar array only when irradiance crosses a baseline threshold of $300\text{ W/m}^2$—then we can eliminate solar drop-offs and optimize clean energy utilization.We estimate this data-driven automation will increase localized solar consumption by 20% on high-yield days and decrease peak-load operational errors by 15%, saving money and reducing the facility's carbon footprint automatically.


## 🚀 The Data Next Steps
This Python proof of concept (PoC) successfully validates the core conditional switching logic using simulated solar telemetry. To elevate this from a scrappy prototype into a production-ready feature request, our next product-data discovery phase must address:
- Look-Ahead Predictive Switching: Relying only on real-time irradiance is a lagging indicator. If a storm is moving in, a sudden drop to $0\text{ W/m}^2$ will shock the system. We need to integrate predictive weather APIs (e.g., 15-minute ahead cloud-cover forecasts) to smoothly ramp down solar utilization before the drop occurs.

- Battery State of Charge (SoC) Integration: Irradiance is only half the equation. Our switching logic must be bi-variate. If solar irradiance is $250\text{ W/m}^2$ (below our threshold) but our onsite battery storage is at $95\%$ capacity, we should still pull from our clean reserves instead of the grid. We need to map out a multi-variable logic matrix with the engineering team.

- Hardware Edge Deployment Strategy: Heavy industrial microgrids cannot risk losing automation if their internet connection drops. We must collaborate with embedded systems engineers to determine if this logic can run locally on an edge computing device (IoT gateway) using compiled, lightweight logic, rather than relying on a cloud-hosted infrastructure.


In [ ]:
import plotly.graph_objects as go
import random

In [ ]:
# 1. PoC Data: Simulating API response from a weather provoider (Solar Irradiance in W/m2)
hours = list(range(24))
solar_irradiance = [max(0,800 * (1-((h-12)/6)**2)) + random.randint(-50,50) for h in hours]

poc_df = pl.DataFrame({'hours':hours,'solar_irradiance_wm2':solar_irradiance}, strict=False)

In [ ]:
# 2. Decision Logic Core: Switch to solar if irradiance > 300 W/m2

poc_df = poc_df.with_columns(
    pl.when(pl.col('solar_irradiance_wm2') > 300)
    .then(pl.lit('Solar Array'))
    .otherwise(pl.lit('Main Grid'))
    .alias('optimal_source')
)

# 3. Preoduct Visual: Presenting the PoC to the engineering lead
fig_1 = go.Figure()
fig_1.add_trace(go.Scatter(x=poc_df['hours'],mode='lines+markers',name='Solar Availabilty'))

# Adding a visual threshold line
fig_1.add_hline(y=300, line_dash='dash', annotation_text='Minimum Solar Threshold')
fig_1.update_layout(title='PoC: Microgrid Power Source Optimization Logic',xaxis_title='Hour of Day',yaxis_title='W/m2')
fig_1.show()

# **Product Specification & Data Exploration: Machine Learning Feature Drivers for Solar Efficiency**

## 📋 Context & Domain
- Product: Solar Asset Performance Management (APM) Software

- Feature: Predictive Maintenance & Efficiency Optimization Module

- Target User: Renewable Energy Asset Managers and Solar Field Maintenance Supervisors


## 🔍 The User Problem
Solar asset managers face an invisible problem: **unexplained degradation**. Over time, solar panels lose their power conversion efficiency due to environmental factors. If efficiency drops too low, the field loses profitability; however, deploying cleaning crews or maintenance technicians prematurely wastes precious operational capital.


Currently, our core software tracks output degradation but cannot pinpoint why a specific string of panels is underperforming. Maintenance teams are left guessing whether a drop in power is caused by normal panel aging, high ambient heat, or heavy dust and particulate accumulation (soiling). Without knowing the root cause, they cannot calculate the exact return on investment (ROI) of sending out a washing truck versus replacing a degrading hardware component.


## 💡 The Product Hypothesis
**If we use a linear regression model to isolate and quantify the specific impacts of ambient temperature versus dust accumulation ($g/\text{m}^2$) on efficiency loss, then we can give asset managers an automated "When to Clean" recommendation engine.**

We believe that by proving dust accumulation has a statistically significant, linear relationship with efficiency loss, we can justify building an automated ROI calculator that triggers washing schedules only when the cost of efficiency loss exceeds the cost of a cleaning crew.


## The Data Next Steps

 This Python machine learning exploration successfully establishes feature coefficients on clean sample telemetry. To transition these findings into a concrete product requirement for our engineering and hardware data pipelines, our next data discovery sprint must address:

 - Sensor Telemetry Expansion: Our current physical hardware stack only measures ambient temperature. This model proves that dust accumulation ($g/\text{m}^2$) is a massive driver of efficiency loss. As a product requirement, we must evaluate whether to integrate local optical soiling sensors into our next hardware lifecycle or pull localized particulate matter data ($PM_{2.5}$ / $PM_{10}$) from public weather and environmental APIs.

 - Addressing Model Non-Linearity: While a simple linear regression gives us quick, explainable baseline coefficients for technical communication, real-world solar degradation is non-linear. High temperatures cause exponential performance degradation (thermal clipping) past certain thresholds ($>35^\circ\text{C}$). We need to work with our data scientists to test polynomial features or random forest models to capture these operational realities accurately.

 - Baseline Calibration (Degradation vs. Soiling): We must separate permanent hardware degradation (aging silicon) from temporary environmental degradation (dust that can be washed away). The product will need historical baseline data for each asset type so the model can accurately isolate a clean, post-rain baseline from a fundamentally decaying panel.




In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# 1. Mocking Clean IoT sensor data

product_data = pl.DataFrame({
    'dust_g_m2': [1.2, 2.5, 3.1, 4.8, 5.0, 6.2, 7.5, 8.1, 9.0, 10.4],
    'ambient_temp_c': [22, 25, 24, 28, 30, 31, 35, 34, 38, 40],
    'efficiency_loss_pct': [0.5, 1.1, 1.4, 2.3, 2.5, 3.1, 3.8, 4.0, 4.7, 5.5]
})

In [ ]:
# 2. Fit a basic linear regression model to find feature weights

X = product_data[['dust_g_m2','ambient_temp_c']].to_numpy()
y = product_data['efficiency_loss_pct'].to_numpy()

model = LinearRegression().fit(X,y)

In [ ]:
# 3. DPM Insight: Translate coefficients for the team
print(f'Product Requirement Insight:')
print(f'-> Every 1g/m2 of dust increases efficiency loss by {model.coef_[0]: .2f}%')
print(f'-> Every 1°C inscrease in temperature increases loss by {model.coef_[1]:.2f}%')

Product Requirement Insight:
-> Every 1g/m2 of dust increases efficiency loss by  0.44%
-> Every 1°C inscrease in temperature increases loss by 0.05%


In [ ]:
# 4. Showing data correlations to stakeholders

fig_2 = px.scatter(
    product_data.to_pandas(),
    x='dust_g_m2',
    y='efficiency_loss_pct',
    color='ambient_temp_c',
    title='ML Feature Exploration: Solar Efficiency Loss Drivers',
    labels={'dust_g_m2':'Dust Accumation (g/m2)','efficiency_loss_pct':'Efficiency Loss (%)'}
)

fig_2.show()

# **Product Specification & Automation Routine: Fleet EV Charging Station Utilization Analytics**

## 📋 Context & Domain
- Product: Municipal & Enterprise EV Fleet Charging Network Management Panel

- Feature: Automated Weekly Utilization & Energy Throughput Accounting

- Target User: Operations Directors, Fleet Mobility Managers, and Regional Sustainability Heads


## 🔍 The User Problem

Managing a growing infrastructure of Electric Vehicle (EV) charging stations introduces a massive operational headache: fragmented, non-standardized telemetry logs. Every week, different charging station hardware vendors drop raw transaction CSV files into storage buckets.

Currently, our Operations Directors spend roughly 3 to 4 hours every Monday morning manually downloading these files, cleaning up missing fields, calculating session durations, and building Excel pivot tables just to report basic North Star metrics—like total kilowatt-hours ($kWh$) delivered and per-station utilization rates. Because this process is manual:

- It is highly prone to human data entry errors.

- The business only receives critical financial performance indicators seven days after the fact, making it impossible to diagnose a failing or underperforming charging station in real time.

- Engineering resources are wasted manually pulling reports for the executive team instead of building platform features.

## 💡 The Product Hypothesis
**If we automate the ingestion, cleansing, and multi-dimensional aggregation of raw charging telemetry using a standardized Polars pipeline, then we can eliminate manual reporting overhead entirely and deliver instant, weekly business-intelligence visual handoffs.**

We expect this automated script to reduce operational reporting time from 4 hours to 0 hours per **week**, guarantee zero computational errors in financial energy metrics, and provide immediate visual clarity on which physical charging nodes are yielding the highest ROI.

## 🚀 The Data Next Steps

This script successfully automates the local cleansing and aggregation logic using Polars expressions. To integrate this routine into our core production cloud microservices, our next product data roadmap must define:
- Handling Missing and Corrupted Hardware Packages: Charging stations occasionally lose cellular connectivity mid-session, resulting in truncated logs (e.g., missing end times or null $kWh$ values). We need to work with data engineers to implement an automated validation gateway (such as Great Expectations) that flags and quarantines corrupted hardware payloads without breaking the main processing pipeline.

- Transitioning to Event-Driven Cloud Triggers: Running a local script is the first checkpoint. Next, we must define the cloud infrastructure requirements to turn this into an event-driven workflow (e.g., triggering an AWS Lambda function running this Polars logic the exact moment a vendor drops a new log into our Amazon S3 bucket).

- Upstream Database Ingestion Schema: Instead of exporting results to flat files, we need to design the star-schema database tables where these aggregated metrics will live. This will allow downstream Business Intelligence (BI) tools (like Tableau or our native customer dashboard) to fetch the optimized metrics instantly through a highly performant API endpoint.

In [ ]:
# 1. Simulated dirty, raw charging log csv data

raw_logs = pl.DataFrame({
    'session_id': [101, 102, 103, 104, 105],
    'station_id': ["STATION_A", "STATION_B", "STATION_A", "STATION_C", "STATION_B"],
    'kwh_delivered': [22.5, 45.1, 12.8, 85.0, 33.2],
    'duration_minutes': [45, 90, 30, 180, 75]
})

In [ ]:
# 2. Automated Aggregatioin Routine
# Calculate efficiency metric (KWh per minute) and total volume per station

automated_report = raw_logs.group_by('station_id').agg([
    pl.col('kwh_delivered').sum().alias('total_kwh_delivered'),
    pl.col('duration_minutes').sum().alias('total_active_minutes'),
    (pl.col('kwh_delivered').sum() / pl.col('duration_minutes').sum()).round(2).alias('kwh_per_minute')
])

In [ ]:
# 3. Export to csv automatically for downstream BI tools
automated_report.write_csv('weekly_ev_utilization_report.csv')

In [ ]:
# 4. Generate the weekly automted performance chart

fig_report = px.bar(
    automated_report.to_pandas(),
    x='station_id',
    y='total_kwh_delivered',
    title='Automated Weekly Metric Report: Enegery Delivered per Station',
    color='station_id'
)

fig_report.update_layout(showlegend=False)
fig_report.show()